## Aggregating (Average) SM data into Monthly Scale

In [ ]:
import glob
import os
import re
import pandas as pd
import xarray as xr
import rioxarray as rxr

# 1. Define the input folder containing your SM GeoTIFFs and the output folder 
#    for monthly averages.
input_folder = r"./data/raw/Soil_Moisture/SM180/South"
output_folder = r"./data/processed/Soil_Moisture/Monthly/South/SM180"
os.makedirs(output_folder, exist_ok=True)

# 2. Gather all TIFF files.
tif_files = sorted(glob.glob(os.path.join(input_folder, "*.tif")))

# 3. Function to extract date from filename.
#    Assumes filenames like: SM_YYYY-MM-DD.tif. Modify if needed.
def extract_date_from_filename(fname):
    base = os.path.basename(fname)
    match = re.search(r"(\d{4}-\d{2}-\d{2})", base)
    if match:
        return pd.to_datetime(match.group(1))
    else:
        print(f"Warning: No valid date found in filename {fname}")
        return None

# 4. Read all rasters into a single xarray DataArray with 'time' as a dimension.
da_list = []
for f in tif_files:
    date = extract_date_from_filename(f)
    if date is None:
        print(f"Skipping file {f} because date could not be parsed.")
        continue
    # Read the raster (masked=True handles nodata), remove extra dimensions, and add a time dimension
    da = rxr.open_rasterio(f, masked=True).squeeze()
    da = da.expand_dims(time=[date])
    da_list.append(da)

if not da_list:
    raise ValueError("No valid rasters were found or parsed. Check your filenames.")

combined_da = xr.concat(da_list, dim="time")
combined_da = combined_da.sortby("time")  # Ensure chronological order

# 5. Assign a new coordinate 'month' using a string representation of year-month.
combined_da = combined_da.assign_coords(month=combined_da.time.dt.strftime("%Y-%m"))

# 6. Group by the 'month' coordinate (only months with data will be processed) 
#    and compute the monthly average.
for month_str, group in combined_da.groupby("month"):
    # Compute the mean across all images in this month
    monthly_mean = group.mean(dim="time")
    
    # Convert the month string to a datetime (using the first day of the month) for filename formatting
    timestamp = pd.to_datetime(month_str + "-01")
    t_str = timestamp.strftime("%Y_%b")
    out_name = os.path.join(output_folder, f"SM180_{t_str}.tif")
    
    # Squeeze to remove any extra dimensions (e.g., the now redundant time dimension)
    monthly_mean = monthly_mean.squeeze()
    
    # Save the monthly average raster to GeoTIFF
    monthly_mean.rio.to_raster(out_name)
    print(f"Saved monthly average for {t_str} to {out_name}")

print("Monthly aggregation complete!")

## Resampling to 5 m and Pixel Alignment with DEM

In [ ]:
import os
import glob
from osgeo import gdal

# ------------------------------------------------------------------------------
# 1. Specify your DEM path (including filename)
# ------------------------------------------------------------------------------
dem_path = r"./data/raw/Topographic_Features/DEM.tif"

# ------------------------------------------------------------------------------
# 2. Specify your ETa imagery folder
#    (change the path to wherever your SM .tif files are stored)
# ------------------------------------------------------------------------------
SM_dir = r"./data/processed/Soil_Moisture/Monthly_10m/South/SM180"

# ------------------------------------------------------------------------------
# 3. Specify the output directory for your resampled ETa files
# ------------------------------------------------------------------------------
out_dir = r"./data/processed/Soil_Moisture/Resampled_10m/South/SM180"
os.makedirs(out_dir, exist_ok=True)

SM_files = glob.glob(os.path.join(SM_dir, "*.tif"))

dem_ds = gdal.Open(dem_path)
dem_proj = dem_ds.GetProjection()  # Get DEM’s projection
dem_ds = None  # Close DEM

for SM_file in SM_files:
    out_file = os.path.join(out_dir, os.path.basename(SM_file))

    warp_options = gdal.WarpOptions(
        format='GTiff',
        xRes=10,               # Desired resolution in X direction
        yRes=10,               # Desired resolution in Y direction
        dstSRS=dem_proj,      # Match the DEM’s projection
        resampleAlg='near',   # Resampling method = nearest neighbor
        targetAlignedPixels=True
    )

    gdal.Warp(
        destNameOrDestDS=out_file,
        srcDSOrSrcDSTab=SM_file,
        options=warp_options
    )

print("Resampling complete!")

### Resampling to 5 m and Pixel Alignment with DEM for SSM

In [ ]:
import os
import glob
from osgeo import gdal

# ------------------------------------------------------------------------------
# 1. Specify your DEM path (including filename)
# ------------------------------------------------------------------------------
dem_path = r"./data/raw/Topographic_Features/akronadjusted_corrected.tif"

# ------------------------------------------------------------------------------
# 2. Specify your ETa imagery folder
#    (change the path to wherever your SM .tif files are stored)
# ------------------------------------------------------------------------------
SM_dir = r"./data/raw/Soil_Moisture/SM30_Raster_MonthlyMean"

# ------------------------------------------------------------------------------
# 3. Specify the output directory for your resampled ETa files
# ------------------------------------------------------------------------------
out_dir = r"./data/processed/Soil_Moisture/Resampled_5m/Monthly"
os.makedirs(out_dir, exist_ok=True)

SM_files = glob.glob(os.path.join(SM_dir, "*.tif"))

dem_ds = gdal.Open(dem_path)
dem_proj = dem_ds.GetProjection()  # Get DEM’s projection
dem_ds = None  # Close DEM

for SM_file in SM_files:
    out_file = os.path.join(out_dir, os.path.basename(SM_file))

    warp_options = gdal.WarpOptions(
        format='GTiff',
        xRes=10,               # Desired resolution in X direction
        yRes=10,               # Desired resolution in Y direction
        dstSRS=dem_proj,      # Match the DEM’s projection
        resampleAlg='near',   # Resampling method = nearest neighbor
        targetAlignedPixels=True
    )

    gdal.Warp(
        destNameOrDestDS=out_file,
        srcDSOrSrcDSTab=SM_file,
        options=warp_options
    )

print("Resampling complete!")

### For growing season

In [ ]:
import os
import glob
from osgeo import gdal

# ------------------------------------------------------------------------------
# 1. Specify your DEM path (including filename)
# ------------------------------------------------------------------------------
dem_path = r"./data/raw/Topographic_Features/akronadjusted_corrected.tif"

# ------------------------------------------------------------------------------
# 2. Specify your ETa imagery folder
#    (change the path to wherever your SM .tif files are stored)
# ------------------------------------------------------------------------------
SM_dir = r"./data/raw/Soil_Moisture/SM30_Raster_GrowingSeasonMean"

# ------------------------------------------------------------------------------
# 3. Specify the output directory for your resampled ETa files
# ------------------------------------------------------------------------------
out_dir = r"./data/processed/Soil_Moisture/Resampled_5m/Growing_Season"
os.makedirs(out_dir, exist_ok=True)

SM_files = glob.glob(os.path.join(SM_dir, "*.tif"))

dem_ds = gdal.Open(dem_path)
dem_proj = dem_ds.GetProjection()  # Get DEM’s projection
dem_ds = None  # Close DEM

for SM_file in SM_files:
    out_file = os.path.join(out_dir, os.path.basename(SM_file))

    warp_options = gdal.WarpOptions(
        format='GTiff',
        xRes=10,               # Desired resolution in X direction
        yRes=10,               # Desired resolution in Y direction
        dstSRS=dem_proj,      # Match the DEM’s projection
        resampleAlg='near',   # Resampling method = nearest neighbor
        targetAlignedPixels=True
    )

    gdal.Warp(
        destNameOrDestDS=out_file,
        srcDSOrSrcDSTab=SM_file,
        options=warp_options
    )

print("Resampling complete!")